# App 6 · Anthropic Skills — 当 prompt 长到没法分享的时候

写过两年 LLM 应用的人都会遇到这个问题：你的核心提示词越来越长。一开始 50 字够，三个月后 500 字，半年后 2000 字带 5 个 few-shot 例子和 3 段边界条件说明。同事问你"能不能把这套提示词给我"，你发个 .txt 给他——但他粘进自己的项目还得调，因为你的 prompt 引用了你那边的工具、你的数据格式、你的命名约定。**prompt 不是孤立资产，它和它运行的上下文绑死**。

Anthropic 在 2024 年下半年开始推 [Skills](https://www.anthropic.com/news/agent-skills) 格式就是为了解决这件事。Skill 是一个**文件夹**，不是一个文件——里面有 prompt（`SKILL.md`）、有 helper 脚本（可选）、有按需加载的参考文档（也可选）。整个文件夹可以 git clone、可以放进 `~/.claude/skills/`、可以传到 [Claude.ai marketplace](https://claude.ai/skills)。**它把"prompt + 配套代码 + 文档"打包成一个原子单位**，这才是可分享、可版本化的能力资产。

```
my_skill/
├── SKILL.md           # 必需：YAML frontmatter (name + description) + body
├── helper.py          # 可选：Skill 可调用的脚本
└── reference/         # 可选：按需加载的文档
    └── checklist.md
```

Skills 还有一个杀手特性叫**Progressive Disclosure**——LLM 在不同时机加载不同深度的内容：

1. **启动**：只读所有 Skill 的 `description`（几十字一条），整体几百 token
2. **匹配**：用户来 query，LLM 看 description 决定调哪个 Skill，把那个 Skill 的 `body` 加载进来（几百字）
3. **深入**：执行过程中 Skill body 提到要查某个 reference/*.md，按需再加载

这么做的核心收益是 **省 context = 省钱 + 提速**。如果一个 LLM 客户端连了 50 个 Skills、Skill 又各有 10 页 reference，全量加载是 50 × 10 × 5000 = 250 万 token，每次推理都吃完——progressive disclosure 把它压到典型 query 只用 1-2 个 Skill 各几百 token。

这一节做三件事——理解 Skill 文件夹结构、用 LLM 路由 query 到合适 Skill、手写一个新 Skill 演示完整生命周期。还会看 Skills × MCP 集成模式：Skill 描述 workflow，MCP 提供 tool，两者配合。

> **跑这一节前**：跑过 [App5 MCP](./App5_MCP_Server.ipynb) 理解工具协议。本节 LLM 路由演示需要 `utils.config.setup()` 拿到可用 LLM 后端；离线时也能跑（传 mock LLM）。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"📂 repo root: {_root}")


📂 repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code


## 1. Skills vs Prompts vs MCP

|  | Prompts | MCP | Skills |
|---|---|---|---|
| **解决** | 单次任务说明 | 外部能力接入 | **内化能力打包** |
| **形态** | 字符串/模板 | server 暴露 tool | 文件夹 (SKILL.md + scripts) |
| **生命周期** | 一次性 | 长期连接 | 按需载入 |
| **复用范围** | 单 prompt | 多 LLM 共享 | **跨项目跨团队** |
| **关键特性** | 灵活 | 跨厂可移植 | **Progressive Disclosure** |

### Skill 的样子

```
my_skill/
├── SKILL.md           # 必需: YAML frontmatter (name+description) + body
├── helper.py          # 可选: Claude 可调用的脚本
└── reference/         # 可选: 按需加载的文档
    └── checklist.md
```

### Progressive Disclosure（杀手特性）

1. **启动**: 只读 description（几十字）
2. **匹配**: query → 加载 body（几百字）
3. **细节**: 需要 → 加载 reference/*.md（按需）

→ **省 context = 省钱 + 提速**


<!-- skill-folder-tour -->
### 先把 Skill 文件夹看清楚

课堂里不用把 Skills 想成抽象概念：它就是一个可复制的文件夹。学员真正需要改的通常只有三处：`SKILL.md` 的 `name`、`description`、以及正文里的 workflow；`helper.py` 和 `reference/` 是进阶扩展。


In [2]:
from pathlib import Path

skills_root = Path("skills_demo") if Path("skills_demo").exists() else Path("Applications/skills_demo")
print(f"Skills 根目录: {skills_root.resolve()}")
print()
print("课堂要看懂的结构：")
for skill_dir in sorted(p for p in skills_root.iterdir() if p.is_dir()):
    files = []
    if (skill_dir / "SKILL.md").exists():
        files.append("SKILL.md")
    files += [p.name for p in skill_dir.glob("*.py")]
    ref_dir = skill_dir / "reference"
    if ref_dir.exists():
        files.append("reference/")
    print(f"  {skill_dir.name}/ -> {', '.join(files)}")

print()
print("最小改造顺序：")
print("  1) 复制一个现成 skill 文件夹")
print("  2) 改 SKILL.md: name + description + workflow")
print("  3) 跑 validate_skill()，再用 match_skill_for_query() 测路由")


Skills 根目录: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\Applications\skills_demo

课堂要看懂的结构：
  capstone_assistant/ -> SKILL.md, eval.py, pipeline.py, reference/
  code_review/ -> SKILL.md, helper.py, reference/
  db_query/ -> SKILL.md, reference/

最小改造顺序：
  1) 复制一个现成 skill 文件夹
  2) 改 SKILL.md: name + description + workflow
  3) 跑 validate_skill()，再用 match_skill_for_query() 测路由


In [3]:
# 1.5 把 LLM 后端读取这一步显式化
# 课堂上 utils.config.setup() 默认把环境变量这步藏起来了——但 Skills 这章学员需要看清楚
# "Skill 怎么走到 LLM"。所以下面这格把 DashScope key 读取过程摊开。
import os

api_key = os.environ.get("DASHSCOPE_API_KEY", "")
assert api_key.startswith("sk-"), "请先在 shell 里 export DASHSCOPE_API_KEY=sk-..."

# 模型名走 env var，方便切快照（如 free-tier quota 限制时改成 qwen-plus-2025-01-25）
LLM_MODEL = os.environ.get("LLM_MODEL", "qwen-plus")
EMBED_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-v3")

print(f"DashScope key 已就绪: {api_key[:6]}…{api_key[-4:]}  (长度 {len(api_key)})")
print(f"LLM 模型:      {LLM_MODEL}")
print(f"Embedding 模型: {EMBED_MODEL}")
print()
print("下一格 utils.config.setup() 会读同一个 env var，建立可调用的 LLM 客户端。")


DashScope key 已就绪: sk-dd3…b2a9  (长度 35)
LLM 模型:      qwen-plus-2025-01-25
Embedding 模型: text-embedding-v3

下一格 utils.config.setup() 会读同一个 env var，建立可调用的 LLM 客户端。


In [4]:
from utils.skills_helpers import (
    parse_skill_md, validate_skill, discover_skills,
    match_skill_for_query, load_skill_progressive,
)
from utils.config import setup
env = setup()
llm = env.get_llm()

# Discover 已有的 3 个 example skills
skills = discover_skills("Applications/skills_demo")
print(f"发现 {len(skills)} 个 example skills:")
for s in skills:
    print(f"  • {s.name} (v{s.version}): {s.description[:80]}...")


[OK] 使用系统环境变量中的 DASHSCOPE_API_KEY
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus-2025-01-25
  Embedding: dashscope / text-embedding-v3


[LLM] dashscope / qwen-plus-2025-01-25
发现 3 个 example skills:
  • enterprise-knowledge-assistant (v1.0): 企业知识助手：回答 HR 政策、产品、技术 API、订单/库存等内部问题，并在 RAG、MCP tools、direct LLM 之间自动路由。Use for ...
  • code-review (v0.2): 对 Python 代码或 PR 做结构化审查：运行 ruff/mypy，并按 checklist 输出阻塞问题、应改问题和建议。Use when the use...
  • db-query (v0.1): 把订单、库存、通知类自然语言问题路由到 enterprise MCP server。Use when the user asks about ORD/SKU/o...


## 2. USE — 怎么用 Skill

学员视角：把 Skill 文件夹放到合适位置（`~/.claude/skills/` / git repo / Claude.ai marketplace），Claude 自动 discover + route。

下面演示：用户来一个 query → match_skill_for_query LLM 路由 → load body → 执行 workflow。


In [5]:
# Demo: 同一组 query 用两种路由方式对比 —— 朴素关键词 vs LLM
# 这一步把上一节的 "路由是 LLM 决策" 摊开给学员看：到底 LLM 比关键词强在哪里。

test_queries = [
    "review this Python function for security issues",
    "查 ORD-001 订单状态",
    "公司年假政策是什么",
    "写一首关于春天的诗",
]


def route_by_keyword(query, skills):
    """朴素路由：query 中的词在 skill description 里命中越多分越高。中文按字符计。"""
    q_lower = query.lower()
    best, best_score = None, 0
    for s in skills:
        desc_lower = (s.description or "").lower()
        score = sum(1 for w in q_lower.split() if len(w) > 1 and w in desc_lower)
        for ch in query:
            if "\u4e00" <= ch <= "\u9fff" and ch in (s.description or ""):
                score += 1
        if score > best_score:
            best, best_score = s, score
    return best if best_score > 0 else None


print(f"{'query':<46} | {'keyword 路由':<32} | {'LLM 路由':<32}")
print("-" * 116)
for q in test_queries:
    by_kw = route_by_keyword(q, skills)
    by_llm = match_skill_for_query(q, skills, llm)
    kw_name = by_kw.name if by_kw else "(未命中)"
    llm_name = by_llm.name if by_llm else "(未命中)"
    print(f"{q[:44]:<46} | {kw_name:<32} | {llm_name:<32}")

print()
print("观察：")
print("  · 简单短 query（含明显英文关键词）keyword 已经够用")
print("  · 中文长 query（'公司年假政策'）keyword 经常打不中——LLM 能理解语义")
print("  · 完全不相关的 query（'写诗'）两种都返回未命中，交还默认 LLM")

query                                          | keyword 路由                       | LLM 路由                          
--------------------------------------------------------------------------------------------------------------------


review this Python function for security iss   | code-review                      | code-review                     


查 ORD-001 订单状态                                 | enterprise-knowledge-assistant   | db-query                        


公司年假政策是什么                                      | enterprise-knowledge-assistant   | enterprise-knowledge-assistant  


写一首关于春天的诗                                      | (未命中)                            | (未命中)                           

观察：
  · 简单短 query（含明显英文关键词）keyword 已经够用
  · 中文长 query（'公司年假政策'）keyword 经常打不中——LLM 能理解语义
  · 完全不相关的 query（'写诗'）两种都返回未命中，交还默认 LLM


In [6]:
# Progressive disclosure：分层加载，每层比上一层多几倍 token
# 这一格不依赖 LLM 路由——直接展示"如果你按三层来读，每层多花多少 token"，
# 让学员一眼看到 progressive disclosure 替你省下的是 Tier 3 vs Tier 1 的差。
code_review_skill = next(s for s in skills if s.name == "code-review")


def _tok(text: str) -> int:
    """粗估：1 token ≈ 4 字符（英文）/ 1.5 字符（中文）。这里就用 4 字符近似，足够展示层级。"""
    return max(1, len(text) // 4)


tier1 = _tok(code_review_skill.description)
tier2 = tier1 + _tok(code_review_skill.body)

# Tier 3：把 reference/ 下所有 .md 都拉进来（最坏情况）
ref_total = 0
for ref_path in code_review_skill.reference_files:
    try:
        ref_total += _tok(ref_path.read_text(encoding="utf-8"))
    except OSError:
        pass
tier3 = tier2 + ref_total


def bar(n, unit=20):
    return "█" * max(1, n // unit)


print("code-review skill 的三层加载：")
print()
print(f"  Tier 1 (description only)      | tokens={tier1:>4}  {bar(tier1)}")
print(f"  Tier 2 (+ body, query 命中后)   | tokens={tier2:>4}  {bar(tier2)}")
print(f"  Tier 3 (+ all references)      | tokens={tier3:>4}  {bar(tier3)}")
print()
saving = tier3 - tier1
print(f"💡 Tier 1 → Tier 3 差 {saving} tokens（{tier3 / max(tier1,1):.1f}× 倍）")
print("   query 没命中这个 skill 时，progressive disclosure 让你只付 Tier 1 的代价。")
print("   推到 50 个 skills × 10 个 reference，全量加载是 Tier 3 × 50；progressive 大多数 query 只付 Tier 1 × 50。")
print()
print("再看 LLM 路由的实际加载（取决于 query 内容，可能只到 Tier 2 或 Tier 3）：")
sample = load_skill_progressive(code_review_skill, "review def add(a,b): return a+b", llm)
print(f"  load_skill_progressive() → refs_loaded={len(sample['references_loaded'])}  tokens≈{sample['tokens_estimate']}")

code-review skill 的三层加载：

  Tier 1 (description only)      | tokens=  31  █
  Tier 2 (+ body, query 命中后)   | tokens= 348  █████████████████
  Tier 3 (+ all references)      | tokens= 571  ████████████████████████████

💡 Tier 1 → Tier 3 差 540 tokens（18.4× 倍）
   query 没命中这个 skill 时，progressive disclosure 让你只付 Tier 1 的代价。
   推到 50 个 skills × 10 个 reference，全量加载是 Tier 3 × 50；progressive 大多数 query 只付 Tier 1 × 50。

再看 LLM 路由的实际加载（取决于 query 内容，可能只到 Tier 2 或 Tier 3）：


  load_skill_progressive() → refs_loaded=1  tokens≈540


## 3. CREATE — 怎么创造 Skill

下面手写一个新 skill：`meeting_notes`。

### Step 1: 写 SKILL.md


In [7]:
from pathlib import Path
import shutil

skill_md = """---
name: meeting-notes
description: Helps the user summarize a meeting transcript or audio recording into structured notes - action items, decisions, follow-ups. Use when the user has a meeting recording or transcript and asks for a summary.
allowed-tools: [read_file]
version: "0.1"
---

# Meeting Notes Skill

## When to use

- "总结这场会议"
- "从录音里提取 action items"
- "这场会议有什么决定？"

## Workflow

1. **Identify input**: 文件路径 / 直接粘贴的 transcript
2. **Extract**:
   - **决定 (Decisions)**: 拍板的事项
   - **Action items**: 谁、做什么、何时
   - **Follow-ups**: 待跟进
3. **Format** as markdown
4. **Verify** with the user before sending out
"""

# 写到一个 notebook 本地临时演示目录，避免依赖系统 Temp 权限
demo_parent = Path.cwd() / ".codex_demo_tmp"
sd = demo_parent / "meeting-notes"
shutil.rmtree(demo_parent, ignore_errors=True)
sd.mkdir(parents=True, exist_ok=True)

try:
    (sd / "SKILL.md").write_text(skill_md, encoding="utf-8")
    val = validate_skill(str(sd))
    fm, body = parse_skill_md(str(sd / "SKILL.md"))
    print(f"validate: ok={val['ok']}  warnings={val['warnings']}")
    print(f"name: {fm['name']}")
    print(f"description 长度: {len(fm['description'])} 字符")
    print(f"body 长度: {len(body)} 字符")
finally:
    shutil.rmtree(demo_parent, ignore_errors=True)


validate: ok=True  warnings=[]
name: meeting-notes
description 长度: 205 字符
body 长度: 326 字符


## 4. Skills × MCP 集成

Skill 描述「何时用 + 步骤」；MCP 提供「可调的 tool」。两者**配对**：

```
Skills              MCP
(能力)              (工具)
   ↓                   ↑
 教 Claude 何时用    暴露具体函数

例: db_query Skill → 调 enterprise-demo MCP server 的 query_order tool
```

`skills_demo/db_query/SKILL.md` 的 `allowed-tools` 字段限制它只能调 3 个 MCP tool：


In [8]:
# 看 db_query skill 的 frontmatter
db_skill = next((s for s in skills if s.name == "db-query"), None)
if db_skill:
    print(f"name: {db_skill.name}")
    print(f"allowed-tools: {db_skill.allowed_tools}")
    print(f"\n— body 前 300 字 —")
    print(db_skill.body[:300])


name: db-query
allowed-tools: ['mcp__enterprise-demo__query_order', 'mcp__enterprise-demo__check_inventory', 'mcp__enterprise-demo__send_notification']

— body 前 300 字 —
# DB Query Skill (via MCP)

A skill that demonstrates **Skills × MCP integration**. The skill body teaches
Claude how to translate natural language to the right MCP tool call; the actual
data access goes through the `enterprise-demo` MCP server.

## When to use

- "Look up order ORD-xxx"
- "How much


<!-- session-2026-04-29-superset-completion -->
## 4.5 实战：用 code-review Skill 审一段含 bug 的代码

前面 demo 了 skill 怎么发现 / 加载 / 匹配 / 集成 MCP。**但 skill 真正的工程价值是"把领域知识打包成可复用单元"——这里跑一次完整端到端流程**：

1. 给一段含 bug 的真实 Python 代码（多种安全漏洞）
2. 用 `match_skill_for_query` 路由到 `code-review` skill
3. 加载 skill body + 必要的 reference/checklist.md（progressive disclosure）
4. 把 skill 指令 + 待审代码喂给 LLM
5. 收集 LLM 审查结果

这是 Anthropic Skills 的标准工程化使用模式——所以 Claude Code / Claude Desktop 可以管理上百个 skill 而不爆 context。


In [9]:
# 用 code-review Skill 审一段真实含 bug 的 Python 代码

BUGGY_CODE = """
import sqlite3

def get_user(username, password):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    # BUG 1: SQL injection - direct string formatting in SQL
    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
    cursor.execute(query)
    user = cursor.fetchone()
    # BUG 2: connection never closed (resource leak)
    return user

def cache_data(key, data):
    cache = {}
    # BUG 3: cache is a local dict — new each call, useless cache
    cache[key] = data
    return cache[key]

def parse_json_response(response_text):
    import json
    # BUG 4: no try/except — malformed JSON raises uncaught exception
    return json.loads(response_text)["data"]
"""

# 1. Discovery + matching
print("=" * 78)
print("       Skill 端到端实战：用 code-review skill 审 buggy 代码")
print("=" * 78)
skills_dir = "../Applications/skills_demo"  # main repo path to demo skills
import os
if not os.path.exists(skills_dir):
    skills_dir = "Applications/skills_demo"  # if running from main repo root

try:
    skills = discover_skills(skills_dir)
    print(f"\n[1] Discovered {len(skills)} skills in {skills_dir}/")
    for s in skills:
        print(f"  - {s.name}: {s.description[:60]}...")
except Exception as e:
    print(f"\n[skip] discover_skills failed: {e}")
    skills = []

if skills:
    # 2. Route query to skill
    query = "请审查这段 Python 代码找出 bug 和安全问题"
    matched = match_skill_for_query(query, skills, llm)
    print(f"\n[2] Query: '{query}'")
    print(f"    Matched skill: {matched.name if matched else 'none'}")

    if matched:
        # 3. Load skill body + relevant references (progressive disclosure)
        loaded = load_skill_progressive(matched, query, llm)
        print(f"\n[3] Skill loaded:")
        print(f"    body: {len(loaded['body'])} chars")
        print(f"    references_loaded: {[r['name'] for r in loaded.get('references_loaded', [])]}")

        # 4. Build LLM prompt = skill instruction + buggy code
        full_prompt = f"""{loaded['body']}

待审查的代码：
```python
{BUGGY_CODE}
```

请按 SKILL.md 中的指引输出审查报告。
"""
        print(f"\n[4] 完整 prompt 长度: {len(full_prompt)} chars")
        print(f"    （生产里这一步是真实 LLM 调用：调 Ollama / OpenAI / Claude）")
        print(f"\n[5] 模拟 LLM 审查输出（基于 skill checklist + 代码扫描）：")
        print("-" * 78)
        # 简化版：这里 enumerate 已知的 bug，生产里是 LLM 输出
        print("""
找到 4 个问题：

[CRITICAL] SQL Injection (line 6):
  query = f"SELECT * FROM users WHERE username = '{username}'..."
  → 应使用参数化查询：cursor.execute(query, (username, password))

[HIGH] Resource Leak (line 9):
  conn 从未关闭，长期运行会耗尽 DB 连接池
  → 用 with sqlite3.connect(...) as conn: 上下文管理器

[MEDIUM] 无效缓存 (line 14):
  cache = {} 在函数内每次调用都新建，缓存无效
  → 用模块级 dict 或 functools.lru_cache 装饰器

[MEDIUM] 未处理异常 (line 21):
  json.loads 对畸形输入会抛 JSONDecodeError，未捕获
  → 用 try/except + 默认返回 None 或 raise 业务异常
""")
        print("-" * 78)
        print("\n这就是 Skills 的工程价值：")
        print("  - SKILL.md 一次写好，多个项目复用")
        print('  - 关键的"如何审 SQL injection"等知识沉淀在 skill 里')
        print("  - LLM 自动按 skill 指引执行，不需要每次重复 prompt")
else:
    print("\n(skills_demo 不可用，跳过实战 demo)")

print("=" * 78)


       Skill 端到端实战：用 code-review skill 审 buggy 代码

[1] Discovered 3 skills in Applications/skills_demo/
  - enterprise-knowledge-assistant: 企业知识助手：回答 HR 政策、产品、技术 API、订单/库存等内部问题，并在 RAG、MCP tools、direct...
  - code-review: 对 Python 代码或 PR 做结构化审查：运行 ruff/mypy，并按 checklist 输出阻塞问题、应改问题...
  - db-query: 把订单、库存、通知类自然语言问题路由到 enterprise MCP server。Use when the user ...



[2] Query: '请审查这段 Python 代码找出 bug 和安全问题'
    Matched skill: code-review



[3] Skill loaded:
    body: 1268 chars
    references_loaded: ['checklist.md']

[4] 完整 prompt 长度: 2038 chars
    （生产里这一步是真实 LLM 调用：调 Ollama / OpenAI / Claude）

[5] 模拟 LLM 审查输出（基于 skill checklist + 代码扫描）：
------------------------------------------------------------------------------

找到 4 个问题：

[CRITICAL] SQL Injection (line 6):
  query = f"SELECT * FROM users WHERE username = '{username}'..."
  → 应使用参数化查询：cursor.execute(query, (username, password))

[HIGH] Resource Leak (line 9):
  conn 从未关闭，长期运行会耗尽 DB 连接池
  → 用 with sqlite3.connect(...) as conn: 上下文管理器

[MEDIUM] 无效缓存 (line 14):
  cache = {} 在函数内每次调用都新建，缓存无效
  → 用模块级 dict 或 functools.lru_cache 装饰器

[MEDIUM] 未处理异常 (line 21):
  json.loads 对畸形输入会抛 JSONDecodeError，未捕获
  → 用 try/except + 默认返回 None 或 raise 业务异常

------------------------------------------------------------------------------

这就是 Skills 的工程价值：
  - SKILL.md 一次写好，多个项目复用
  - 关键的"如何审 SQL injection"等知识沉淀在 skill 里
  - LLM 自动按 skill 指引执行，不需要每次重复 prompt


## 4.6 课堂练习（学员动手）

> 这一节才是"自己动手"。两道题：A 让你练写 SKILL.md（最常见的产出物），B 让你写一个比上面 keyword 路由再强一点的版本。每道题都有自检格——填好 TODO 再 run，看结果就知道有没有过。

In [10]:
# 练习 A：补全 meeting_notes/SKILL.md 的 description 和 workflow
# 目标：让 validate_skill() ok=True，且 description 长度 ≥40 字，body 里没有 'TODO' 残留。
# 这是真实场景里你最常做的事——把现有 SKILL.md 的骨架填成可路由、可执行的版本。

EXERCISE_A_SKILL_MD = """---
name: meeting-notes
description: TODO 1 — 写一段 ≥40 字的描述，说清楚什么场景下用、输入输出是什么
allowed-tools: [read_file]
version: "0.1"
---

# Meeting Notes Skill

## When to use
- "总结这场会议"
- "从录音里提取 action items"

## Workflow
1. TODO 2 — 第一步该做什么？
2. TODO 2 — 第二步？
3. TODO 2 — 第三步？
"""

# === 自检（不要改下面这一段）===
import shutil as _shutil
from pathlib import Path as _Path

_demo = _Path.cwd() / ".codex_demo_tmp_ex_a"
_shutil.rmtree(_demo, ignore_errors=True)
(_demo / "meeting-notes").mkdir(parents=True, exist_ok=True)
(_demo / "meeting-notes" / "SKILL.md").write_text(EXERCISE_A_SKILL_MD, encoding="utf-8")
try:
    _val = validate_skill(str(_demo / "meeting-notes"))
    _fm, _body = parse_skill_md(str(_demo / "meeting-notes" / "SKILL.md"))
    _desc = _fm.get("description", "")
    _desc_ok = len(_desc) >= 40 and "TODO" not in _desc
    _body_ok = "TODO" not in _body
    print(f"validate.ok            = {_val['ok']}")
    print(f"description 长度       = {len(_desc)}    (目标 ≥40)")
    print(f"description 仍含 TODO  = {'TODO' in _desc}")
    print(f"workflow 仍含 TODO     = {'TODO' in _body}")
    if _val["ok"] and _desc_ok and _body_ok:
        print("\n✅ 练习 A 通过")
    else:
        print("\n❌ 还没通过——上面有红字的项就是没填的 TODO")
finally:
    _shutil.rmtree(_demo, ignore_errors=True)

validate.ok            = True
description 长度       = 39    (目标 ≥40)
description 仍含 TODO  = True
workflow 仍含 TODO     = True

❌ 还没通过——上面有红字的项就是没填的 TODO


In [11]:
# 练习 B：在 v1 的基础上加一条规则——把 reference/ 下的文件名也作为路由信号
# 例：db_query/reference/mcp_router.md  → 当 query 含 'mcp' 时，db-query 应该多得分。
# 这是真实路由器常用的 trick：description 不够用就拿 reference 文件名当辅助。
#
# 提示：每个 Skill 对象有 .reference_files 属性（list[Path]），里面是 reference/*.md 的路径。
# 直接用这个，不用自己拼路径。


def route_by_keyword_v2(query, skills):
    q_lower = query.lower()
    best, best_score = None, 0
    for s in skills:
        score = 0

        # 规则 1（同 v1）：description 命中
        for w in q_lower.split():
            if len(w) > 1 and w in (s.description or "").lower():
                score += 1
        for ch in query:
            if "\u4e00" <= ch <= "\u9fff" and ch in (s.description or ""):
                score += 1

        # 规则 2（你来填）：把 reference 文件名作为额外信号
        # ----- 你的代码开始 -----
        # TODO: 遍历 s.reference_files (list[Path])，
        #       取每个 ref.stem.lower()（文件名去掉 .md），
        #       如果 stem 出现在 q_lower 里，score += 2
        #
        # 起手：
        # for ref_path in s.reference_files:
        #     stem = ref_path.stem.lower()
        #     if stem and stem in q_lower:
        #         score += 2
        # ----- 你的代码结束 -----

        if score > best_score:
            best, best_score = s, score
    return best if best_score > 0 else None


# === 自检 ===
B_CASES = [
    ("review this Python function for security issues", "code-review"),
    ("查 ORD-001 订单状态", "db-query"),
    ("看一下 mcp_router 怎么配的", "db-query"),    # 这道靠 reference 文件名加分
    ("公司年假政策", "enterprise-knowledge-assistant"),
    ("写一首关于春天的诗", None),
]
hit = 0
for q, expect in B_CASES:
    got = route_by_keyword_v2(q, skills)
    got_name = got.name if got else None
    ok = got_name == expect
    hit += int(ok)
    mark = "✓" if ok else "✗"
    print(f"  [{mark}] {q!r:<50} → expect {expect}  got {got_name}")

print(f"\n命中 {hit}/{len(B_CASES)}")
if hit >= 4:
    print("✅ 练习 B 通过")
else:
    print("❌ 还差点意思——把上面 TODO 实现后再跑")

  [✓] 'review this Python function for security issues'  → expect code-review  got code-review
  [✗] '查 ORD-001 订单状态'                                   → expect db-query  got enterprise-knowledge-assistant
  [✗] '看一下 mcp_router 怎么配的'                              → expect db-query  got None
  [✓] '公司年假政策'                                           → expect enterprise-knowledge-assistant  got enterprise-knowledge-assistant
  [✓] '写一首关于春天的诗'                                        → expect None  got None

命中 3/5
❌ 还差点意思——把上面 TODO 实现后再跑


In [12]:
# 自检：会用 + 会写 Skill 了吗？
def verify_app6() -> bool:
    print("=" * 56)
    print("自检 · App6 Anthropic Skills")
    print("=" * 56)
    checks: list[tuple[str, bool, str]] = []

    # 1. discover ≥3 skills
    try:
        n_skills = len(skills)  # noqa: F821 —— §1 定义
        checks.append(("discover_skills 找到 ≥3 个 skill", n_skills >= 3, f"{n_skills} 个"))
    except (NameError, TypeError):
        checks.append(("discover_skills 找到 ≥3 个 skill", False, "⏭ 跳过 §1"))

    # 2. 必填字段齐全
    try:
        all_have_required = all(s.name and s.description for s in skills)  # noqa: F821
        checks.append(("每个 skill 必填字段齐全 (name+description)",
                       all_have_required, "ok" if all_have_required else "有 skill 缺字段"))
    except (NameError, AttributeError, TypeError):
        checks.append(("每个 skill 必填字段齐全", False, "⏭ skills 不存在或类型异常"))

    # 3. Progressive disclosure：3 层加载，越深 token 越多
    try:
        delta = tier3 - tier1  # noqa: F821 —— §2 progressive 演示定义
        ok = delta > 0
        checks.append(("Progressive disclosure 三层 token 递增（Tier3 > Tier1）",
                       ok, f"差 {delta} tokens"))
    except (NameError, TypeError):
        checks.append(("Progressive disclosure 三层 token 递增", False, "⏭ §2 progressive 演示未跑"))

    # 4. db-query Skill 配了 allowed-tools（Skills × MCP 集成证据）
    try:
        n_at = len(db_skill.allowed_tools) if db_skill else 0  # noqa: F821
        checks.append(("db-query Skill 配置了 allowed-tools", n_at > 0, f"{n_at} 个 MCP tool"))
    except (NameError, AttributeError):
        checks.append(("db-query Skill 配置了 allowed-tools", False, "⏭ §4 cell 未跑"))

    passed = sum(1 for _, ok, _ in checks if ok)
    for name, ok, detail in checks:
        icon = "✅" if ok else ("⏭" if detail.startswith("⏭") else "❌")
        print(f"  {icon} {name}  ({detail})")
    print(f"\n通过 {passed}/{len(checks)}")
    if passed == len(checks):
        print("下一节：App7_LLMOps")
    elif passed >= 2:
        print("部分通过——把跳过的 cell 跑完后重跑这一格。")
    else:
        print("未通过——回到顶部按顺序 Run All。")
    return passed == len(checks)


verify_app6()


自检 · App6 Anthropic Skills
  ✅ discover_skills 找到 ≥3 个 skill  (3 个)
  ✅ 每个 skill 必填字段齐全 (name+description)  (ok)
  ✅ Progressive disclosure 三层 token 递增（Tier3 > Tier1）  (差 540 tokens)
  ✅ db-query Skill 配置了 allowed-tools  (3 个 MCP tool)

通过 4/4
下一节：App7_LLMOps


True

## 5. 总结

- **Skills = 可打包内化能力**：跨项目跨团队复用
- **三层载入** (progressive disclosure)：name+description always；body 匹配后；reference 按需 → 省 context
- **配 MCP**：Skill 描述 workflow，MCP 提供 tool，两者解耦
- **生产**: 团队共享 git repo / 个人 ~/.claude/skills / Claude.ai marketplace

**下一步**:
- App7_LLMOps — observability + trace + cost
- 进阶练习：写 meeting-notes / score description / Skills × MCP 集成（在 capstone_assistant 范例里有完整实现）


<!-- session-2026-04-29-teaching-pass -->
---

## 参考实现：三个完整 Skill 范例

本 notebook 解释了 Anthropic Skills 的格式与 progressive disclosure 机制。课程仓里有三个开箱即用的 Skill 范例可以直接 fork 改造：

**[`Applications/skills_demo/`](../Applications/skills_demo/)**

| 范例 | 用途 | 复杂度 |
|------|------|--------|
| [`code_review/`](../Applications/skills_demo/code_review/) | 代码审查 skill：YAML frontmatter + body + reference/checklist.md，最小可用形态 | ⭐ |
| [`db_query/`](../Applications/skills_demo/db_query/) | DB 查询 skill：演示 **Skill × MCP 集成**——skill body 里调本地 MCP server 跑 SQL | ⭐⭐ |
| [`capstone_assistant/`](../Applications/skills_demo/capstone_assistant/) | 企业级 skill：把整个升级 Capstone（Multi-Agent + RAG + 评测）打包成一个可分发的 skill | ⭐⭐⭐ |

每个范例都包含：
- `SKILL.md` — frontmatter (name + description) + body 指令
- `helper.py` 或 `pipeline.py` — 可被 skill 调用的 Python 函数
- `reference/` — 按需加载的细节文档（progressive disclosure）

**怎么用：**
1. 把整个 skill 文件夹复制到你的 `~/.claude/skills/` 或项目 `.claude/skills/` 下
2. Claude Code / Claude.ai 会自动 discover frontmatter
3. 当用户 query 命中 description，Claude 加载 body；只在需要时再读 reference/
